# Solution Intelligence Engine

This notebook walks through the core flow: load the synthetic knowledge
dataset, run it through the ingestion pipeline (Stage 1 structural check,
AI quality judge, duplicate check), then retrieve proven solutions for a
new support issue.

In [ ]:
from solution_intelligence.service import SolutionEngine
from solution_intelligence.sources import load_filtered

engine = SolutionEngine()
entries = load_filtered(max_total=12)
run = engine.ingest(entries)

print(f"Ingested: {len(run.ingested)}  Rejected: {len(run.rejected)}")
print(f"Knowledge index size: {engine.index.size}")

In [ ]:
results = engine.search(
    "SAP FI report access denied for new user - authorization error",
    top_k=3,
)

for i, r in enumerate(results, start=1):
    e = r.entry
    print(f"{i}. {e.title}")
    print(
        f"   match={r.combined_score:.0%}  confidence={r.confidence:.0%}  "
        f"worked {e.worked}/{e.attempted} times"
    )
    print(f"   resolution: {e.resolution[:80]}...")
    print()

In [ ]:
# Conversational refinement demo
session = engine.agent.start("demo", "VPN drops after 5 minutes on Windows 11")
initial_top = session.turns[-1].candidates[0]
print(f"Initial top match: {initial_top.entry.title}")

session = engine.agent.respond("demo", "also check large file transfer cases")
refined_top = session.turns[-1].candidates[0]
print(f"After refinement:  {refined_top.entry.title}")